# I. Tarmoq Arxitekturasi va Soketlar Anatomiyasi

---

### 1.1. OSI modeli vs TCP/IP: Encapsulation (Ma'lumotning paketlanishi)

Tarmoq orqali ma'lumot uzatish jarayoni **Encapsulation** (qobiqqa o'rash) deb ataladi. Tasavvur qiling, siz xat yuboryapsiz: xatni konvertga solasiz, ustiga manzil yozasiz va markasini yopishtirasiz. Tarmoqda ham xuddi shunday.

* **OSI (Open Systems Interconnection) modeli:** 7 qatlamdan iborat nazariy model.
* **TCP/IP modeli:** 4 qatlamdan iborat amaliy model (internet shu asosda ishlaydi).

**Encapsulation jarayoni:**
1.  **Application (Ilova):** Sizning dasturingiz (masalan, "Salom" xabari).
2.  **Transport (TCP/UDP):** Xabarga **Port** raqamlari qo'shiladi. Ma'lumot **Segment** deb ataladi.
3.  **Internet (IP):** Segmentga **IP-manzillar** qo'shiladi. Endi u **Paket** deb ataladi.
4.  **Network Access:** Paketga **MAC-manzil** qo'shiladi. Endi u **Freym** (Frame) deb ataladi va bitlar ko'rinishida sim orqali ketadi.

<div align="center">
    <img src="images/OSI vs TCP_IP.jpg" width="500px">
    <img src="images/OSI-vs-TCP-vs-Hybrid-2.webp" width="800px">
</div>

---

### 1.2. Soket tushunchasi va Identifikatsiya

**Soket** — bu dastur va tarmoq o'rtasidagi interfeys. Operatsion tizim nuqtai nazaridan soket bu **Fayl deskriptori** (File Descriptor) hisoblanadi. Ya'ni, Python-da fayl bilan qanday ishlasangiz (ochish, yozish, yopish), soket bilan ham shunday ishlaysiz.

#### IP-manzillar (Manzil)
* **IPv4:** 32-bitli manzil (masalan, `192.168.1.1`). Maksimal $2^{32}$ ta manzil bor.
* **IPv6:** 128-bitli manzil (masalan, `2001:0db8:85a3...`). Manzillar yetishmovchiligi muammosini hal qiladi.

#### Portlar klassifikatsiyasi (Eshiklar)
Port — bu kompyuter ichidagi qaysi dasturga ulanishni belgilovchi raqam (0 dan 65535 gacha):
* **System Ports (0-1023):** Standart xizmatlar (80: HTTP, 443: HTTPS, 22: SSH).
* **User/Registered Ports (1024-49151):** Maxsus dasturlar (masalan, 3306: MySQL).
* **Dynamic/Private Ports (49152-65535):** Vaqtinchalik ulanishlar uchun.

Soketni (Socket) tushunish uchun uni dasturlash va apparat ta'minoti (hardware) o'rtasidagi **"ko'prik"** yoki **"oxirgi nuqta"** deb tasavvur qilish kerak.

Operatsion tizim (Windows, Linux, macOS) uchun soket xuddi **fayl** kabi ko'rinadi. Siz faylni ochasiz, unga ma'lumot yozasiz (`write`) va undan ma'lumot o'qiysiz (`read`). Tarmoqda ham xuddi shunday: soketga ma'lumot yozasiz, u tarmoq orqali ikkinchi tomonga yetib boradi.

Soketning tarkibiy qismlari (Manzil)
Bitta soketni tarmoqda tanib olish uchun 5 ta element kerak (bunga **5-tuple** deyiladi):
1.  **Protokol:** (TCP yoki UDP).
2.  **Lokal IP-manzil:** Ma'lumot chiqayotgan kompyuter manzili.
3.  **Lokal Port:** Ma'lumot chiqayotgan dasturning "eshigi".
4.  **Masofaviy IP-manzil:** Ma'lumot boradigan kompyuter manzili.
5.  **Masofaviy Port:** Ma'lumot boradigan dasturning "eshigi".

Soket turlari (Protokolga ko'ra)
Eng ko'p qo'llaniladigan ikki tur:

* **Stream Sockets (SOCK_STREAM / TCP):**
    * **Xususiyati:** Ishonchli. Ma'lumotlar xuddi oqim (stream) kabi tartib bilan boradi. Agar yo'lda paket yo'qolsa, TCP uni qaytadan yuboradi.
    * **Misol:** HTTP (veb-saytlar), Email, Fayl yuklash.
* **Datagram Sockets (SOCK_DGRAM / UDP):**
    * **Xususiyati:** Tezkor, lekin kafolatsiz. Ma'lumot bo'laklari (paketlar) tartibsiz borishi yoki yo'qolishi mumkin.
    * **Misol:** Video qo'ng'iroqlar, Onlayn o'yinlar (ping kam bo'lishi uchun).

Fayl Deskriptori sifatida Soket
Siz Python-da `socket.socket()` funksiyasini chaqirganingizda, operatsion tizim sizga bitta butun son (masalan, `3` yoki `540`) qaytaradi. Bu son **File Descriptor** deyiladi.
* Operatsion tizim ichida bitta jadval bor.
* O'sha jadvalda `3`-raqamli soket qaysi IP va qaysi Portga bog'langani yozilgan bo'ladi.
* Siz `send()` buyrug'ini berganingizda, OS o'sha jadvalga qarab ma'lumotni qayerga haydashni bilib oladi.

Soketning hayotiy sikli (Server misolida)
Server soketi doimo quyidagi bosqichlardan o'tadi:
1.  **Creation (Yaratish):** "Menga aloqa vositasi kerak".
2.  **Binding (Bog'lash):** "Men 8080-portda va shu IP-da ishlayman".
3.  **Listening (Eshitish):** "Kimdir ulanishini kutib turibman".
4.  **Acceptance (Qabul):** "Yangi mijoz keldi, u bilan gaplashish uchun alohida kanal ochdim".



---

Kiberxavfsizlik nuqtai nazaridan soketlar

Soketlarni tushunish xavfsizlik uchun juda muhim:
* **Port Exhaustion:** Agar siz ulanishlarni (soketlarni) to'g'ri yopmasangiz (`.close()`), tizimda bo'sh portlar qolmaydi va yangi ulanishlarni qabul qilolmay qoladi. Bu "Denial of Service" (DoS) holatiga sabab bo'ladi.
* **Raw Sockets:** Ba'zi maxsus soketlar paketning hamma qismini (hatto IP-headerini ham) qo'lda yozishga imkon beradi. Buzg'unchilar bundan **IP Spoofing** (o'zini boshqa IP qilib ko'rsatish) uchun foydalanishadi.
* **Socket Hijacking:** Ishlab turgan soket ulanishini o'g'irlash orqali ma'lumotlarni qo'lga kiritish mumkin.

---

### 1.3. TCP Protokoli: Aloqa o'rnatish va uzish

TCP — bu ulanishga asoslangan (**connection-oriented**) protokol. Ma'lumot uzatishdan oldin ikki tomon "kelishib" oladi.

#### Uch bosqichli salomlashish (3-way Handshake)
1.  **SYN (Synchronize):** Klient serverga ulanish so'rovini yuboradi.
2.  **SYN-ACK:** Server so'rovni qabul qiladi va o'zining tayyorligini bildiradi.
3.  **ACK (Acknowledge):** Klient ulanish o'rnatilganini tasdiqlaydi.

<div align="center">
    <img src="images/TCP_connection.jpg" width="500px">
</div>

#### Ulanishni uzish (4-way Termination)
TCP aloqani uzishda har bir tomon mustaqil ravishda o'z qismini yopishi kerak:
1.  Klient **FIN** yuboradi (Men boshqa ma'lumot yubormayman).
2.  Server **ACK** yuboradi (Tushundim).
3.  Server ham o'z ishini tugatgach, **FIN** yuboradi.
4.  Klient **ACK** yuboradi va aloqa butunlay yopiladi.


<div align="center">
    <img src="images/TCP terminate.png" width="500px">
</div>

---

### 1.4. Kiberxavfsizlik asoslari

Tarmoq dasturchisi sifatida siz quyidagi xavf va metodlarni bilishingiz shart:

* **Sniffing (Paketlarni ushlash):** Tarmoq trafigini kuzatish. Agar ma'lumot shifrlanmagan bo'lsa (HTTP, FTP kabi), buzg'unchi `Wireshark` kabi vositalar bilan foydalanuvchi parollarini o'qib olishi mumkin.
    * *Himoya:* SSL/TLS shifrlashdan foydalanish.
* **Port Scanning:** Buzg'unchi nishon kompyuterga turli portlar bo'yicha SYN paketlarini yuborib ko'radi. Agar javob kelsa, demak u yerda qandaydir dastur ishlayapti va bu dasturda zaiflik bo'lishi mumkin.
    * *Himoya:* Firewall (brandmauer) orqali keraksiz portlarni yopish va `rate limiting` (ulanishlar sonini cheklash) o'rnatish.

---
